# Checking yields for group3 with details per category

In [1]:
%run load-rds-final-2012-23903000-rs.ipynb

Welcome to JupyROOT 6.28/00


Loaded rdf with 305828 enries


In [2]:
def mygroupby(d, groupbycols):
    g = d.groupby(groupbycols).size().reset_index(name='count').sort_values([ 'count'], ascending=False).reset_index(drop=True)
    g["Percentage"] = g.apply(lambda row: 100 * row["count"]/d.shape[0], axis=1)
    g["cumulative %"] = g["Percentage"].cumsum(axis = 0)
    return g

In [3]:
def categ_groupby(df, groupbycol, threshold):
    """ Group the dataframe df by column groupbycol, grouping together entries rows with count < threshold """
    g = pd.DataFrame(df.groupby(groupbycol).count()["category"]).rename(columns={"category":"count"})
    g["Category"] = g.apply(lambda row: row.name if row["count"] > threshold else "others", axis=1)
    g2 = g.groupby("Category").sum()
    g2 = g2.sort_values([ 'count'], ascending=False)
    total = g2.sum()["count"]
    g2["Percentage"] = g2.apply(lambda row: 100 * row["count"]/total, axis=1)
    g2["cumulative %"] = g2["Percentage"].cumsum(axis = 0)
    return g, g2

In [4]:
#%load_ext autoreload
#%autoreload
import categories4 as f
rdf_initial = rdf
rdf = f.add_categories_and_filter(rdf_initial,  apply_BDT_Iso_cut=True, apply_PIDK_cut=True)

In [5]:
base_columns = [ "eventIndex", "category", "simplified", "B_M", "B_Y_SEP", "Xc_signal_Ypis_displaced_fromBs_fromTau", "fromY_from_B_vertex", "BDT_Iso", "q2_2", "tauY_2"]
load_columns = [
    "Y_0_40_nc_mult",
    "Y_0_20_cc_mult",
    "Y_0_20_cc_PZ",
    "Y_0_30_nc_PZ",
    "Y_0_40_nc_PZ",
    "min_m2pi",
    "max_m2pi",
    "missing_mass_2",
    "B_BPVVDR",
    "B_M",
    "B_correctedMass",
#     "log(abs(PBsn))",
#     "log(abs(PBv/B_P))",
#     "log(abs(PBvn/B_P))",
#     "log(abs((PBsn-PBvn)/PBvn))",
#     "log(sqrt(abs(mDs2vn)))",
    "mN2v",
#    "log(Y_PE)",
    "BDT_Iso",
    "B_pT_Bdir",
    "Y_BPVVDR",
    "missing_pY_mass",
    "Y_correctedMass",
    'PBsn',
    'PBv', 'PBvn', 'B_P', 'mDs2vn', 'Y_PE',
    "Xc_fromDs",
    "Xc_fromDps1",
    "Xc_fromDsst0",
    "Xc_fromDsst",
    "Y_fromDps1",
    "Y_fromDsst0",
    "Y_fromDsst",
]
train_columns = [
    "Y_0_40_nc_mult",
    "Y_0_20_cc_mult",
    "Y_0_20_cc_PZ",
    "Y_0_30_nc_PZ",
    "Y_0_40_nc_PZ",
    "min_m2pi",
    "max_m2pi",
    "missing_mass_2",
    "B_BPVVDR",
    "B_M",
    "B_correctedMass",
    "log(abs(PBsn))",
    "log(abs(PBv/B_P))",
    "log(abs(PBvn/B_P))",
    "log(abs((PBsn-PBvn)/PBvn))",
    "log(sqrt(abs(mDs2vn)))",
    "mN2v",
    "log(Y_PE)",
    "BDT_Iso",
    "B_pT_Bdir",
    "Y_BPVVDR",
    "missing_pY_mass",
    "Y_correctedMass",
]

columns = base_columns +  load_columns
df = pd.DataFrame(rdf.Cache(columns).AsNumpy())


In [6]:
df['simplified_key'] = df.apply(lambda row: f.pretty_categories_map[row["simplified"]], axis=1)
df['key'] = df.apply(lambda row: f.categories_map[row["category"]], axis=1)
df['signal'] = (df['simplified_key'] == 'Signal')
dfbm = df.query("B_M < 5000")

In [7]:
df.query("B_M < 5000").query("signal == True").query("q2_2 > 0")

,BDT_Iso,B_BPVVDR,B_M,B_P,B_Y_SEP,B_correctedMass,B_pT_Bdir,PBsn,PBv,PBvn,...,max_m2pi,min_m2pi,missing_mass_2,missing_pY_mass,q2_2,simplified,tauY_2,simplified_key,key,signal
18,0.300701,1.251786,3555.436260,73440.504631,-5.396421,4648.655407,964.673579,95030.086006,119258.329865,102598.838899,...,717.463194,326.297603,634.665738,-309.581326,7.492151,1007,0.000324,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
27,0.304464,2.207561,4295.377515,161397.486742,-13.755033,4867.665362,538.646115,218321.801510,227549.612874,218321.949795,...,1136.189727,631.568058,341.499885,317.509062,4.852885,1007,0.000512,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
191,0.399764,3.409034,4101.212991,144294.265574,-29.192873,5547.156730,1257.491041,123290.341564,132639.509066,130663.716106,...,711.771211,547.652312,-1185.865988,536.725748,7.190575,1007,0.000946,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
192,0.060515,1.316995,4603.941355,152274.692578,-17.808110,5027.680017,405.882071,148826.309470,151727.047076,148868.065327,...,873.148779,786.979971,-356.337971,396.317398,5.524680,1007,0.000726,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
274,0.316373,1.963349,4468.346674,191079.648151,-6.288015,5286.581659,754.913492,161847.575966,179169.819382,161867.100541,...,721.087590,655.613555,-754.646648,517.683761,6.000162,1007,0.000149,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242268,0.235340,1.554001,3972.670890,207990.577214,-7.814765,5305.832564,1165.674289,292365.048373,351147.073467,316351.741998,...,932.845543,836.936942,-971.323028,486.019541,3.913299,1007,0.000234,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
242374,0.430140,6.910549,3387.932555,77325.281992,-23.642952,3737.534285,333.251195,81476.991060,88985.847274,81830.132992,...,595.212368,570.175348,852.768024,838.358234,8.128510,1007,0.000633,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
242395,0.198366,2.908379,3819.010212,90789.831007,-7.146156,4266.577350,424.092075,107943.957038,116331.299456,107979.802988,...,891.375999,764.576723,758.678435,529.004453,6.473756,1007,0.000470,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True
242476,0.229566,1.213361,4300.833408,145979.948903,-8.153945,5283.667754,891.424042,127477.921173,131290.652600,127653.882759,...,759.718200,435.177340,-815.592170,463.230325,7.859131,1007,0.000334,Signal,Xc_signal_Ypis_displaced_fromBs_fromTau,True


In [8]:
dfbm.query("BDT_Iso <  0.03458")

,BDT_Iso,B_BPVVDR,B_M,B_P,B_Y_SEP,B_correctedMass,B_pT_Bdir,PBsn,PBv,PBvn,...,max_m2pi,min_m2pi,missing_mass_2,missing_pY_mass,q2_2,simplified,tauY_2,simplified_key,key,signal


THIS INCLUDES THE CUT q2_2 > 0 BY DEFAULT

In [9]:
import joblib
bdtdblcharm = joblib.load("../../train_bdt/bdtdblcharm_150_3_0.04.pkl")
def add_cols_for_bdt(tmpdf):
    df = tmpdf.copy()
    df["log(abs(PBsn))"] = np.log(np.abs(df["PBsn"]))
    df["log(abs(PBv/B_P))"] = np.log(np.abs(df["PBv"] / df["B_P"]))
    df["log(abs(PBvn/B_P))"] = np.log(np.abs(df["PBvn"] / df["B_P"]))
    df["log(abs((PBsn-PBvn)/PBvn))"] = np.log(np.abs((df["PBsn"] - df["PBvn"]) / df["PBvn"]))
    df["log(sqrt(abs(mDs2vn)))"] = np.log(np.sqrt(np.abs(df["mDs2vn"])))
    df["log(Y_PE)"] = np.log(df["Y_PE"]) 
    df["diff_m2pi"] = df["max_m2pi"] - df["min_m2pi"]
    return df
dfbm2 = add_cols_for_bdt(dfbm)

In [10]:
dfbm2['bdt_dc'] = bdtdblcharm.predict_proba(dfbm2[train_columns])[:,1]

In [11]:
dfbm2.query("signal == True")['category'].unique()

array([24, 10, 11], dtype=int32)

In [12]:
f.categories_map

{0: 'Xc_background',
 1: 'Xc_signal_Ypis_diffAncestorYXc',
 2: 'Xc_signal_Ypis_B_vertex_fromBs',
 3: 'Xc_signal_Ypis_B_vertex_fromOtherB',
 4: 'Xc_signal_Ypis_B_vertex_fromHc',
 5: 'Xc_signal_Ypis_B_vertex_fromNone',
 6: 'Xc_signal_Ypis_nomatch_Prompt',
 7: 'Xc_signal_Ypis_nomatch_doubleCharm',
 8: 'Xc_signal_Ypis_nomatch_charmStrange',
 9: 'Xc_signal_Ypis_nomatch_Other',
 10: 'Xc_signal_Ypis_diffVertex_signal',
 11: 'Xc_signal_Ypis_diffVertex_tauFromB',
 12: 'Xc_signal_Ypis_diffVertex_normlike',
 13: 'Xc_signal_Ypis_diffVertex_doubleCharm',
 14: 'Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB',
 15: 'Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB',
 16: 'Xc_signal_Ypis_diffVertex_CharmStrange',
 17: 'Xc_signal_Ypis_diffVertex_SomeFromPV',
 18: 'Xc_signal_Ypis_displaced_fromBs_fromDs',
 19: 'Xc_signal_Ypis_displaced_fromB0_fromDp',
 20: 'Xc_signal_Ypis_displaced_fromBp_fromD0',
 21: 'Xc_signal_Ypis_displaced_fromLambdab_fromLambdac',
 22: 'Xc_signal_Ypis_displaced_fromBs_fromDp',
 23: 

In [13]:
dfbm2

,BDT_Iso,B_BPVVDR,B_M,B_P,B_Y_SEP,B_correctedMass,B_pT_Bdir,PBsn,PBv,PBvn,...,key,signal,log(abs(PBsn)),log(abs(PBv/B_P)),log(abs(PBvn/B_P)),log(abs((PBsn-PBvn)/PBvn)),log(sqrt(abs(mDs2vn))),log(Y_PE),diff_m2pi,bdt_dc
0,0.142126,5.122865,3549.868167,168977.117807,-42.580564,4429.095708,791.959042,213931.912377,227255.066624,224798.696179,...,Xc_background,False,12.273413,0.296310,0.285442,-3.029495,7.911052,10.883605,414.693391,0.480915
1,0.084323,1.668125,3319.233948,61299.597437,-25.553933,3650.446180,316.186461,68422.978613,75375.247132,68627.893311,...,Xc_background,False,11.133464,0.206706,0.112926,-5.813861,7.786525,9.734375,197.234971,0.047678
2,0.368573,1.993594,3952.608126,156837.935763,-10.530797,4331.339197,362.173002,193830.018453,206759.162365,194002.880756,...,Xc_signal_Ypis_displaced_fromB0_fromDp,False,12.174737,0.276342,0.212660,-7.023133,7.734093,11.047794,35.228538,0.550376
3,0.152137,0.780185,3590.944305,86795.626730,-5.040866,4689.721792,970.058554,63525.280784,74187.153322,64659.427607,...,Xc_background,False,11.059193,-0.156965,-0.294422,-4.043253,7.936118,10.755483,143.757553,0.744786
4,0.382045,1.299142,3747.756129,148853.385470,-5.497857,4000.589420,244.843884,152223.181078,163115.939253,152258.133512,...,Xc_signal_Ypis_displaced_fromBs_fromDs,False,11.933103,0.091499,0.022615,-8.379345,7.086510,9.932745,56.547468,0.057617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242735,0.321024,4.012284,4027.889398,202833.051607,-20.939083,5027.599310,900.316559,278687.778202,301173.947829,286704.281254,...,Xc_signal_Ypis_displaced_fromBs_fromDs,False,12.537847,0.395305,0.346068,-3.576949,7.392935,11.387447,558.693289,0.197994
242736,0.132013,0.912369,4565.097589,171913.729033,-12.364674,5011.791461,426.787276,169474.374381,172384.198847,169695.757786,...,Xc_signal_Ypis_displaced_fromBs_fromDs,False,12.040457,0.002733,-0.012986,-6.641866,7.637914,10.947215,93.671025,0.435207
242737,0.340379,1.283536,4689.555203,142928.520106,-29.903899,6174.122483,1306.085235,123794.055575,126366.400090,123879.637927,...,Xc_signal_Ypis_displaced_fromBs_fromDs,False,11.726375,-0.123159,-0.143034,-7.277587,7.654822,10.930162,275.300333,0.302139
242738,0.266166,1.156650,3400.800053,108148.088194,-5.197422,4531.720777,989.806372,145142.143730,176817.601762,145184.537346,...,Xc_signal_Ypis_nomatch_doubleCharm,False,11.885469,0.491617,0.294504,-8.138763,7.694428,10.076992,399.626947,0.345274


# Loading the template

In [14]:
import yaml
from yaml import SafeLoader
# Loading the template configuration
with open("../config/templates.yaml") as yf:
    templates_config = yaml.load(yf, Loader=SafeLoader)

In [15]:
group3 = templates_config['templates']['group3']

In [16]:
group3_map = {}
group3_others = None
for i, c in enumerate(group3):
    if type(c) == int:
        group3_map[c] = i
    elif c == 'others':
        group3_others = i
    else:
        for cc in c:
            if cc == 'others':
                group3_others = i
            else:
                group3_map[cc] = i

def get_group3_categories(bkgcat):
    return group3[group3_map.get(bkgcat, group3_others)]

def _g1cat2name(cat):
    if type(cat) == int:
        tmp = f.categories_map[cat]
        if tmp == "Xc_signal_Ypis_displaced_fromBs_fromTau":
            return "Signal"
        elif tmp in ['Xc_signal_Ypis_diffVertex_signal', 'Xc_signal_Ypis_diffVertex_tauFromB']:
            return None
        return tmp.replace("Xc_signal_Ypis_", "")
    elif cat == 'others':
        return 'others'
    elif type(cat) == list:
        #print(cat)
        catnames = [ _g1cat2name(c) for c in cat]
        #print(f"===> {catnames}")
        return ", ".join([cc for cc in catnames if cc is not None])
        #return ", ".join([cc for cc in catnames in cc is not None])
 

def _get_group3_name(c):
    #print(f"==>categ {c}")
    return _g1cat2name(group3[int(c)])

def _get_group3_name_pretty(c):
    #print(f"==>categ {c}")
    n = _g1cat2name(group3[int(c)])
    if "displaced_fromB0_fromD0" in n:
        return "DoubleCharm1"
    if "displaced_fromBs_fromD0" in n:
        return "DoubleCharm2"
    if "displaced_fromB0_fromDs" in n:
        return "DoubleCharm2"    
    if "displaced_fromBs_fromDp" in n:
        return "DoubleCharm2"
    if "displaced_fromBp_fromDp" in n:
        return "DoubleCharm2"
    if "Signal" in n:
        return "Signal"
    if "diffVertex_CharmStrange" in n:
        return "DoubleCharm3"
    if "Xc_background" in n:
        return "BadDs"
    if "displaced_fromBs_fromDs" in n:
        return "DoubleCharm4"
    if "displaced_fromBp_fromD0" in n:
        return "DoubleCharm5"
    if "Others" in n:
        return "others"
    if "displaced_fromLambdab" in n:
        return "Lambdab"
    if "diffVertex_doubleCharm" in n:
        return "DoubleCharm6"
    return n


def _template_name(g):
    prefix = "TemplateCat"
    if type(g) == list:
        return prefix + "_".join([str(gg) for gg in g])
    else:
        return prefix + str(g)
    
for g in group3:
    print(f"{g} -> {[f.categories_map[gg] for gg in g] if type(g) == list else 'others' if g == 'others' else f.categories_map[g] }")

    
cat_mapper = { _g1cat2name(g):_template_name(g)  for g in group3}
cat_mapper['Signal'] = 'Signal'

[24, 10, 11] -> ['Xc_signal_Ypis_displaced_fromBs_fromTau', 'Xc_signal_Ypis_diffVertex_signal', 'Xc_signal_Ypis_diffVertex_tauFromB']
20 -> Xc_signal_Ypis_displaced_fromBp_fromD0
[18, 29] -> ['Xc_signal_Ypis_displaced_fromBs_fromDs', 'Xc_signal_Ypis_displaced_fromBs_fromDs_fromTau']
21 -> Xc_signal_Ypis_displaced_fromLambdab_fromLambdac
[25, 13, 19, 7] -> ['Xc_signal_Ypis_displaced_fromB0_fromD0', 'Xc_signal_Ypis_diffVertex_doubleCharm', 'Xc_signal_Ypis_displaced_fromB0_fromDp', 'Xc_signal_Ypis_nomatch_doubleCharm']
[16, 14] -> ['Xc_signal_Ypis_diffVertex_CharmStrange', 'Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB']
others -> others
[23, 22, 26, 27] -> ['Xc_signal_Ypis_displaced_fromBp_fromDp', 'Xc_signal_Ypis_displaced_fromBs_fromDp', 'Xc_signal_Ypis_displaced_fromB0_fromDs', 'Xc_signal_Ypis_displaced_fromBs_fromD0']
0 -> Xc_background
15 -> Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB


In [17]:
dfbm2['group3_category'] = dfbm2.apply(lambda row: group3_map.get(row['category'], group3_others), axis=1)
dfbm2['group3_name'] = dfbm2.apply(lambda row: _get_group3_name_pretty(row['group3_category']), axis=1)
#dfbm2



In [18]:
pd.set_option('display.max_colwidth', None)
stat_g3 = mygroupby(dfbm2, 'group3_name')

# Evaluating datasets after CUT1

In [19]:
df_cut1 = dfbm2.query("bdt_dc >0.75").copy()
df_cut1

,BDT_Iso,B_BPVVDR,B_M,B_P,B_Y_SEP,B_correctedMass,B_pT_Bdir,PBsn,PBv,PBvn,...,log(abs(PBsn)),log(abs(PBv/B_P)),log(abs(PBvn/B_P)),log(abs((PBsn-PBvn)/PBvn)),log(sqrt(abs(mDs2vn))),log(Y_PE),diff_m2pi,bdt_dc,group3_category,group3_name
18,0.300701,1.251786,3555.436260,73440.504631,-5.396421,4648.655407,964.673579,95030.086006,119258.329865,102598.838899,...,11.461949,0.484816,0.334351,-2.606798,7.925230,10.280254,391.165591,0.835851,0,Signal
39,0.283367,1.292672,3671.926334,73427.704872,-7.681154,5003.469723,1154.365561,110835.043343,140589.916271,112069.435627,...,11.615798,0.649546,0.422817,-4.508540,7.491961,10.208661,480.446961,0.782136,2,DoubleCharm4
42,0.404309,2.684474,3607.369402,102132.771520,-7.656490,4313.122682,648.012320,92406.482514,96050.139020,94066.082585,...,11.433952,-0.061403,-0.082276,-4.037421,7.668874,10.252635,290.374934,0.830894,7,DoubleCharm2
72,0.329701,1.612059,4002.168662,187145.479473,-10.946915,4528.762500,495.978292,200135.963988,208853.464725,201535.767453,...,12.206752,0.109747,0.074081,-4.969635,7.736399,11.134590,383.567002,0.784657,3,Lambdab
73,0.330229,2.266750,3577.040816,78797.771518,-4.658925,4716.835622,1002.082545,105864.626889,127206.967064,106309.222401,...,11.569916,0.478931,0.299467,-5.476942,7.729613,9.824016,252.351130,0.790929,5,DoubleCharm3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242557,0.323769,0.807329,3835.593566,99225.978707,-21.442189,5779.289013,1616.842786,415133.757856,641675.566942,491666.292153,...,12.936356,1.866683,1.600400,-1.860084,8.155133,9.924101,398.874661,0.875915,2,DoubleCharm4
242635,0.284516,0.666451,3158.451203,55726.754664,-7.998039,6315.834374,2368.170860,-88.617160,4114.352658,4558.759608,...,4.484326,-2.605979,-2.503410,0.019252,9.470674,9.646987,240.399349,0.774663,8,BadDs
242637,0.254442,2.124364,3350.702911,66972.296137,-9.963435,5170.482222,1499.538703,22368.108969,23229.919209,23792.096737,...,10.015392,-1.058838,-1.034926,-2.815892,7.956278,10.475185,64.621909,0.848526,6,others
242731,0.333332,2.244371,3774.096978,91924.263631,-17.255412,4524.046993,687.790512,151586.875608,159625.762436,154494.492451,...,11.928914,0.551867,0.519193,-3.972825,7.868195,10.676019,289.118094,0.890382,4,DoubleCharm1


In [45]:
def detailed_categ(df_cut1):
    categ_count = df_cut1[['group3_name']].groupby(['group3_name']).value_counts().copy()
    subcateg_count = df_cut1[['key']].groupby(['key']).value_counts().copy()
    g1 = df_cut1[['group3_name', 'key']].groupby(['group3_name', 'key'])
    sorted_keys = sorted(list(g1.groups.keys()), key=lambda k: 1000 * categ_count[k[0]] + subcateg_count[k[1]], reverse=True)
    total_candidates = len(df_cut1)
    vals = []
    for k in sorted_keys:
        tg = g1.groups[k]
        vals.append([int(len(tg) / 12.5),  100*len(tg)/categ_count[k[0]], int(categ_count[k[0]] / 12.5), categ_count[k[0]]/total_candidates * 100])
    return pd.DataFrame(data=vals, index=pd.MultiIndex.from_tuples(sorted_keys), columns=['yield 2invfb', '% of group', 'group yield 2invfb', 'group % total'])

In [49]:
print(detailed_categ(df_cut1).to_latex(float_format="%.2f",))

\begin{tabular}{llrrrr}
\toprule
             &                                                &  yield 2invfb &  \% of group &  group yield 2invfb &  group \% total \\
\midrule
DoubleCharm1 & Xc\_signal\_Ypis\_displaced\_fromB0\_fromDp &           257 &       79.55 &                 323 &          30.39 \\
             & Xc\_signal\_Ypis\_nomatch\_doubleCharm &            42 &       13.20 &                 323 &          30.39 \\
             & Xc\_signal\_Ypis\_displaced\_fromB0\_fromD0 &            14 &        4.50 &                 323 &          30.39 \\
             & Xc\_signal\_Ypis\_diffVertex\_doubleCharm &             8 &        2.74 &                 323 &          30.39 \\
DoubleCharm2 & Xc\_signal\_Ypis\_displaced\_fromBs\_fromDp &           135 &       63.68 &                 213 &          20.03 \\
             & Xc\_signal\_Ypis\_displaced\_fromBp\_fromDp &            57 &       27.02 &                 213 &          20.03 \\
             & Xc\_signal\_Ypis\_displaced\

/tmp/ipykernel_95171/2978449668.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(detailed_categ(df_cut1).to_latex(float_format="%.2f",))


# Categorisation after CUT2

In [50]:
df_cut2 = dfbm2.query("bdt_dc >0.35").copy()
df_cut2

,BDT_Iso,B_BPVVDR,B_M,B_P,B_Y_SEP,B_correctedMass,B_pT_Bdir,PBsn,PBv,PBvn,...,log(abs(PBsn)),log(abs(PBv/B_P)),log(abs(PBvn/B_P)),log(abs((PBsn-PBvn)/PBvn)),log(sqrt(abs(mDs2vn))),log(Y_PE),diff_m2pi,bdt_dc,group3_category,group3_name
0,0.142126,5.122865,3549.868167,168977.117807,-42.580564,4429.095708,791.959042,213931.912377,227255.066624,224798.696179,...,12.273413,0.296310,0.285442,-3.029495,7.911052,10.883605,414.693391,0.480915,8,BadDs
2,0.368573,1.993594,3952.608126,156837.935763,-10.530797,4331.339197,362.173002,193830.018453,206759.162365,194002.880756,...,12.174737,0.276342,0.212660,-7.023133,7.734093,11.047794,35.228538,0.550376,4,DoubleCharm1
3,0.152137,0.780185,3590.944305,86795.626730,-5.040866,4689.721792,970.058554,63525.280784,74187.153322,64659.427607,...,11.059193,-0.156965,-0.294422,-4.043253,7.936118,10.755483,143.757553,0.744786,8,BadDs
5,0.380439,1.331113,3759.179166,118719.466875,-6.950834,4606.385172,769.296974,148691.858377,187742.944822,148886.256130,...,11.909631,0.458310,0.226419,-6.641032,7.592225,10.733137,140.973504,0.732946,3,Lambdab
8,0.386120,1.268511,4653.164085,131076.783535,-34.190821,4937.144719,275.813465,126677.082164,131028.124198,126891.715626,...,11.749396,-0.000371,-0.032449,-6.382158,7.835766,11.180211,541.115814,0.359199,8,BadDs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242722,0.035130,2.385021,3807.294184,77704.508995,-9.069162,4025.452387,212.246694,79423.650931,84968.769616,79619.730235,...,11.282551,0.089370,0.024349,-6.006498,7.845617,10.172563,120.959594,0.415187,8,BadDs
242728,0.299081,3.732498,3449.593454,191235.470501,-4.900037,5165.208898,1430.696052,104479.382065,152697.037294,152320.477076,...,11.556745,-0.225050,-0.227519,-1.158102,7.870213,11.457336,205.322881,0.353511,6,others
242731,0.333332,2.244371,3774.096978,91924.263631,-17.255412,4524.046993,687.790512,151586.875608,159625.762436,154494.492451,...,11.928914,0.551867,0.519193,-3.972825,7.868195,10.676019,289.118094,0.890382,4,DoubleCharm1
242732,0.301964,1.160974,3878.392064,84065.390206,-5.678718,4390.305486,482.068648,88138.109878,95429.033792,88787.196582,...,11.386660,0.126788,0.054648,-4.918431,7.749765,10.264957,64.443898,0.789360,6,others


In [53]:
detailed_categ(df_cut2)

yield 2invfb  \
DoubleCharm1 Xc_signal_Ypis_displaced_fromB0_fromDp                       1824   
             Xc_signal_Ypis_nomatch_doubleCharm                            387   
             Xc_signal_Ypis_displaced_fromB0_fromD0                        145   
             Xc_signal_Ypis_diffVertex_doubleCharm                          90   
DoubleCharm4 Xc_signal_Ypis_displaced_fromBs_fromDs                        946   
             Xc_signal_Ypis_displaced_fromBs_fromDs_fromTau                 54   
DoubleCharm3 Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB                875   
             Xc_signal_Ypis_diffVertex_CharmStrange                         84   
DoubleCharm2 Xc_signal_Ypis_displaced_fromBs_fromDp                        533   
             Xc_signal_Ypis_displaced_fromBp_fromDp                        273   
             Xc_signal_Ypis_displaced_fromB0_fromDs                         73   
             Xc_signal_Ypis_displaced_fromBs_fromD0                         43   
BadDs        Xc_background                                                 668   
DoubleCharm5 Xc_signal_Ypis_displaced_fromBp_fromD0                        600   
Signal       Xc_signal_Ypis_displaced_fromBs_fromTau                       237   
             Xc_signal_Ypis_diffVertex_signal                                2   
             Xc_signal_Ypis_diffVertex_tauFromB                              0   
Lambdab      Xc_signal_Ypis_displaced_fromLambdab_fromLambdac              193   
others       Xc_signal_Ypis_B_vertex_fromBs                                 41   
             Xc_signal_Ypis_nomatch_Prompt                                  27   
             Xc_signal_Ypis_displaced_fromBp_fromDs                         25   
             Xc_signal_Ypis_nomatch_charmStrange                            18   
             Xc_signal_Ypis_diffAncestorYXc                                 16   
             Xc_signal_Ypis_diffVertex_SomeFromPV                           12   
             Xc_signal_Ypis_displaced_fromLambdab_fromDs                     8   
             Xc_signal_Ypis_displaced_fromLambdab_fromDp                     7   
             Xc_signal_Ypis_displaced_fromB0_fromDs_fromTau                  3   
             Xc_signal_Ypis_B_vertex_fromOtherB                              2   
             Xc_signal_Ypis_displaced_fromB0_fromDp_fromTau                  2   
             Xc_signal_Ypis_displaced_fromXic                                2   
             Xc_signal_Ypis_diffVertex_normlike                              1   
             Xc_signal_Ypis_B_vertex_fromHc                                  1   
             NA                                                              1   
             Xc_signal_Ypis_displaced_fromBs                                 1   
             Xc_signal_Ypis_displaced_fromBp_fromDs_fromTau                  1   
             Xc_signal_Ypis_displaced_fromLambdab_fromD0                     0   
             Xc_signal_Ypis_displaced_fromBs_fromDp_fromTau                  0   
             Xc_signal_Ypis_displaced_fromLambdab_fromDs_fromTau             0   
             Xc_signal_Ypis_nomatch_Other                                    0   
             Xc_signal_Ypis_displaced_fromBp_fromDp_fromTau                  0   
             Xc_signal_Ypis_displaced_NA                                     0   
             Xc_signal_Ypis_displaced_fromB0_fromLambdac                     0   
             Xc_signal_Ypis_displaced_fromLambdac                            0   
             Xc_signal_Ypis_displaced_fromBp                                 0   
DoubleCharm6 Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB                 55   

                                                                  % of group  \
DoubleCharm1 Xc_signal_Ypis_displaced_fromB0_fromDp                74.537431   
             Xc_signal_Ypis_nomatch_doubleCharm                    15.818895   
             Xc_signal_Ypis_displaced_fromB0_fromD0  

In [54]:
print(detailed_categ(df_cut2).to_latex(float_format="%.2f",))

\begin{tabular}{llrrrr}
\toprule
             &                                                &  yield 2invfb &  \% of group &  group yield 2invfb &  group \% total \\
\midrule
DoubleCharm1 & Xc\_signal\_Ypis\_displaced\_fromB0\_fromDp &          1824 &       74.54 &                2447 &          33.67 \\
             & Xc\_signal\_Ypis\_nomatch\_doubleCharm &           387 &       15.82 &                2447 &          33.67 \\
             & Xc\_signal\_Ypis\_displaced\_fromB0\_fromD0 &           145 &        5.96 &                2447 &          33.67 \\
             & Xc\_signal\_Ypis\_diffVertex\_doubleCharm &            90 &        3.68 &                2447 &          33.67 \\
DoubleCharm4 & Xc\_signal\_Ypis\_displaced\_fromBs\_fromDs &           946 &       94.51 &                1001 &          13.78 \\
             & Xc\_signal\_Ypis\_displaced\_fromBs\_fromDs\_fromTau &            54 &        5.49 &                1001 &          13.78 \\
DoubleCharm3 & Xc\_signal\_Ypis\_d

/tmp/ipykernel_95171/325177978.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(detailed_categ(df_cut2).to_latex(float_format="%.2f",))


# Categorisation after CUT3

In [55]:
df_cut3 = dfbm2.query("bdt_dc >0.5").copy()
df_cut3

,BDT_Iso,B_BPVVDR,B_M,B_P,B_Y_SEP,B_correctedMass,B_pT_Bdir,PBsn,PBv,PBvn,...,log(abs(PBsn)),log(abs(PBv/B_P)),log(abs(PBvn/B_P)),log(abs((PBsn-PBvn)/PBvn)),log(sqrt(abs(mDs2vn))),log(Y_PE),diff_m2pi,bdt_dc,group3_category,group3_name
2,0.368573,1.993594,3952.608126,156837.935763,-10.530797,4331.339197,362.173002,193830.018453,206759.162365,194002.880756,...,12.174737,0.276342,0.212660,-7.023133,7.734093,11.047794,35.228538,0.550376,4,DoubleCharm1
3,0.152137,0.780185,3590.944305,86795.626730,-5.040866,4689.721792,970.058554,63525.280784,74187.153322,64659.427607,...,11.059193,-0.156965,-0.294422,-4.043253,7.936118,10.755483,143.757553,0.744786,8,BadDs
5,0.380439,1.331113,3759.179166,118719.466875,-6.950834,4606.385172,769.296974,148691.858377,187742.944822,148886.256130,...,11.909631,0.458310,0.226419,-6.641032,7.592225,10.733137,140.973504,0.732946,3,Lambdab
12,0.329347,0.826890,4597.299860,77469.817714,-9.138854,6996.769070,1988.032701,56182.497376,59301.477418,56220.787654,...,10.936361,-0.267254,-0.320602,-7.291846,7.713983,10.520637,410.193870,0.579836,8,BadDs
13,0.431613,2.167060,4302.201065,107321.622836,-38.769828,5643.233070,1181.693660,121029.988508,138704.219978,131847.404767,...,11.703794,0.256514,0.205815,-2.500488,7.453222,10.930394,491.496389,0.600485,2,DoubleCharm4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242700,0.348936,0.912110,3973.709476,98453.847184,-27.404613,4982.234476,906.450050,154457.697644,172658.631451,156512.541273,...,11.947676,0.561729,0.463548,-4.332936,7.604979,10.723937,710.535329,0.712234,7,DoubleCharm2
242710,0.296153,0.831610,4033.569387,140440.588664,-19.288269,4678.864244,600.796296,149933.550476,158098.091919,152769.656660,...,11.917947,0.118431,0.084147,-3.986499,7.820788,10.932432,182.066864,0.592671,4,DoubleCharm1
242711,0.393721,0.797451,3622.412318,95820.105100,-12.494819,4191.831767,530.744411,90808.029049,93436.280535,90978.910029,...,11.416503,-0.025193,-0.051845,-6.277416,7.698065,10.186167,385.483607,0.731181,7,DoubleCharm2
242731,0.333332,2.244371,3774.096978,91924.263631,-17.255412,4524.046993,687.790512,151586.875608,159625.762436,154494.492451,...,11.928914,0.551867,0.519193,-3.972825,7.868195,10.676019,289.118094,0.890382,4,DoubleCharm1


In [57]:
detailed_categ(df_cut3)

yield 2invfb  \
DoubleCharm1 Xc_signal_Ypis_displaced_fromB0_fromDp                       1098   
             Xc_signal_Ypis_nomatch_doubleCharm                            211   
             Xc_signal_Ypis_displaced_fromB0_fromD0                         83   
             Xc_signal_Ypis_diffVertex_doubleCharm                          48   
DoubleCharm2 Xc_signal_Ypis_displaced_fromBs_fromDp                        373   
             Xc_signal_Ypis_displaced_fromBp_fromDp                        185   
             Xc_signal_Ypis_displaced_fromB0_fromDs                         45   
             Xc_signal_Ypis_displaced_fromBs_fromD0                         27   
DoubleCharm3 Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB                504   
             Xc_signal_Ypis_diffVertex_CharmStrange                         46   
DoubleCharm4 Xc_signal_Ypis_displaced_fromBs_fromDs                        442   
             Xc_signal_Ypis_displaced_fromBs_fromDs_fromTau                 37   
BadDs        Xc_background                                                 412   
DoubleCharm5 Xc_signal_Ypis_displaced_fromBp_fromD0                        326   
Signal       Xc_signal_Ypis_displaced_fromBs_fromTau                       204   
             Xc_signal_Ypis_diffVertex_signal                                2   
             Xc_signal_Ypis_diffVertex_tauFromB                              0   
Lambdab      Xc_signal_Ypis_displaced_fromLambdab_fromLambdac              111   
others       Xc_signal_Ypis_B_vertex_fromBs                                 26   
             Xc_signal_Ypis_nomatch_Prompt                                  18   
             Xc_signal_Ypis_displaced_fromBp_fromDs                         16   
             Xc_signal_Ypis_nomatch_charmStrange                             9   
             Xc_signal_Ypis_diffAncestorYXc                                  7   
             Xc_signal_Ypis_diffVertex_SomeFromPV                            6   
             Xc_signal_Ypis_displaced_fromLambdab_fromDs                     5   
             Xc_signal_Ypis_displaced_fromLambdab_fromDp                     5   
             Xc_signal_Ypis_displaced_fromB0_fromDs_fromTau                  2   
             Xc_signal_Ypis_B_vertex_fromOtherB                              2   
             Xc_signal_Ypis_displaced_fromB0_fromDp_fromTau                  1   
             NA                                                              1   
             Xc_signal_Ypis_displaced_fromXic                                1   
             Xc_signal_Ypis_B_vertex_fromHc                                  1   
             Xc_signal_Ypis_diffVertex_normlike                              0   
             Xc_signal_Ypis_displaced_fromBp_fromDs_fromTau                  0   
             Xc_signal_Ypis_displaced_fromBs_fromDp_fromTau                  0   
             Xc_signal_Ypis_displaced_fromBs                                 0   
             Xc_signal_Ypis_displaced_fromLambdab_fromD0                     0   
             Xc_signal_Ypis_displaced_fromLambdab_fromDs_fromTau             0   
             Xc_signal_Ypis_displaced_fromBp_fromDp_fromTau                  0   
             Xc_signal_Ypis_nomatch_Other                                    0   
             Xc_signal_Ypis_displaced_NA                                     0   
             Xc_signal_Ypis_displaced_fromB0_fromLambdac                     0   
             Xc_signal_Ypis_displaced_fromBp                                 0   
             Xc_signal_Ypis_displaced_fromLambdac                            0   
DoubleCharm6 Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB                 35   

                                                                  % of group  \
DoubleCharm1 Xc_signal_Ypis_displaced_fromB0_fromDp                76.149027   
             Xc_signal_Ypis_nomatch_doubleCharm                    14.675389   
             Xc_signal_Ypis_displaced_fromB0_fromD0  

In [58]:
print(detailed_categ(df_cut3).to_latex(float_format="%.2f",))

\begin{tabular}{llrrrr}
\toprule
             &                                                &  yield 2invfb &  \% of group &  group yield 2invfb &  group \% total \\
\midrule
DoubleCharm1 & Xc\_signal\_Ypis\_displaced\_fromB0\_fromDp &          1098 &       76.15 &                1442 &          33.48 \\
             & Xc\_signal\_Ypis\_nomatch\_doubleCharm &           211 &       14.68 &                1442 &          33.48 \\
             & Xc\_signal\_Ypis\_displaced\_fromB0\_fromD0 &            83 &        5.82 &                1442 &          33.48 \\
             & Xc\_signal\_Ypis\_diffVertex\_doubleCharm &            48 &        3.36 &                1442 &          33.48 \\
DoubleCharm2 & Xc\_signal\_Ypis\_displaced\_fromBs\_fromDp &           373 &       59.00 &                 632 &          14.68 \\
             & Xc\_signal\_Ypis\_displaced\_fromBp\_fromDp &           185 &       29.38 &                 632 &          14.68 \\
             & Xc\_signal\_Ypis\_displaced\

/tmp/ipykernel_95171/1814944491.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(detailed_categ(df_cut3).to_latex(float_format="%.2f",))
